# KD teacher-student với ReviewKD trên Mixture-36l

| Nhánh | Đường đi | Trạng thái |
| --- | --- | --- |
| Teacher | mixture → FRCRN (nhiễu ước lượng, chuẩn hoá RMS) → BEATs (lớp 3, 6, 9, 12) | đóng băng |
| Student | mixture → log-mel → PANNs CNN10 (4 stage) → 36 lớp | huấn luyện từ đầu |
| KD | ReviewKD: ABF hợp nhất stage sâu vào stage nông, HCL (MSE + pyramid 4/2/1) với 4 lớp BEATs | chỉ lúc train |

Cấu hình: `config/reviewkd.yaml`. Mã: `src/model/reviewkd/`, `src/loss/reviewkd.py`, `src/train/reviewkd_trainer.py`,
`scripts/run_reviewkd.py`. Chi tiết toán: `latex/reviewkd/main.tex`.

`RUN_TRAINING = True` chạy thật; `False` chạy smoke test nhỏ vào `output/test/`.
Cell cuối đặt `is_completed = True` chỉ khi `RUN_TRAINING = True` và mọi bước đã xong.

In [ ]:
import json
import os
import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# (1) Tên pipeline, chế độ chạy, đường dẫn đầu ra (RunPaths), logger
from src.utils.logger import setup_logger
from src.utils.paths import RunPaths

PIPELINE_NAME = "reviewkd"
RUN_TRAINING = False  # True: chạy thật; False: smoke test nhỏ ghi vào output/test
RESUME_RUN_ID = None  # ví dụ "reviewkd-2026-10-17-09-00-00": mô hình đã có history.json thì bỏ qua

output_dir = Path("output") if RUN_TRAINING else Path("output") / "test"
if RESUME_RUN_ID:
    run = RunPaths.from_run_id(RESUME_RUN_ID, output_dir)
else:
    run = RunPaths.new(PIPELINE_NAME if RUN_TRAINING else f"{PIPELINE_NAME}-smoke", output_dir)
run.create()

logger = setup_logger(PIPELINE_NAME, run.log_file)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("run %s | device %s | RUN_TRAINING=%s", run.run_id, device, RUN_TRAINING)

In [ ]:
# (2) Tham số cấu hình
NUM_WORKERS = 8
INCLUDE_BASELINE = True  # chạy thêm baseline CE thuần (lambda = 0) để so sánh
USE_DRIVE = False  # True: nếu thiếu pretrained/frcrn_best.pt thì tải từ Google Drive (cần gdrive_fsspec)

In [ ]:
# (3) Nạp dataset: Mixture-36l (train/validation); AudioSet-40l chỉ cần khi target_rms = 0 (tính lại RMS nhiễu)
from scripts.download_dataset import DATASETS, ensure_dataset
from scripts.run_reviewkd import read_reviewkd_config

raw = read_reviewkd_config()
wanted = [raw["dataset"]["name"]]
if not raw["review"]["target_rms"]:
    wanted.append(raw["dataset"]["noise_dataset"])
_missing = [
    key
    for key in wanted
    if not ((Path("datasets") / DATASETS[key].name).is_dir() and any((Path("datasets") / DATASETS[key].name).iterdir()))
]
if _missing and not os.environ.get("KAGGLE_ACCESS_TOKEN"):
    raise RuntimeError(f"Set KAGGLE_ACCESS_TOKEN to download {_missing}")
dataset_roots = {key: ensure_dataset(key, Path("datasets")) for key in wanted}
dataset_roots

In [ ]:
# (4) Pretrain: BEATs thuần từ HuggingFace, FRCRN từ pretrained/ (hoặc Drive); cả hai bị đóng băng khi train
from scripts.load_pretrain import download_frcrn_best, download_pretrained
from scripts.run_reviewkd import resolve_path

beats_path = resolve_path(raw["teacher"]["beats_checkpoint"])
if not beats_path.is_file():
    download_pretrained("beats")
frcrn_path = resolve_path(raw["teacher"]["frcrn_checkpoint"])
if not frcrn_path.is_file():
    if not USE_DRIVE:
        raise FileNotFoundError(f"{frcrn_path} not found: copy FRCRN best.pt there or set USE_DRIVE = True")
    from gdrive_fsspec import GoogleDriveFileSystem

    _fs = GoogleDriveFileSystem(
        use_listings_cache=False, skip_instance_cache=True, auth_kwargs={"use_local_webserver": False}
    )
    download_frcrn_best(_fs, frcrn_path)
print("FRCRN:", frcrn_path, "| BEATs:", beats_path)

In [ ]:
# (5) Kiến trúc: lưới đặc trưng của student và teacher phải khớp, đếm tham số
from scripts.run_baselines import count_millions
from scripts.run_reviewkd import build_review, build_student, check_grids
from src.model.reviewkd import ReviewKDConfig

grid = check_grids(raw)
review_config = ReviewKDConfig.from_dict(raw["review"])
print(
    f"grid (freq, time) {grid} | student {count_millions(build_student(raw)):.2f}M | "
    f"review module {count_millions(build_review(raw, review_config, grid)):.2f}M | "
    f"BEATs layers {list(review_config.teacher_layers)}"
)

In [ ]:
# (6) Huấn luyện: baseline CE (lambda = 0) rồi ReviewKD; mô hình đã có history.json thì bỏ qua
from scripts.run_reviewkd import RESULT_COLUMNS, run_reviewkd

rows = run_reviewkd(
    run, device, logger, include_baseline=INCLUDE_BASELINE, smoke=not RUN_TRAINING, num_workers=NUM_WORKERS
)
for row in rows:
    print({key: row[key] for key in RESULT_COLUMNS})

In [ ]:
# Hoàn thành: is_completed = True khi mọi mô hình đã có history, result.csv đã ghi và đủ dòng kết quả
expected = ["cnn10-reviewkd"] + (["cnn10-ce"] if INCLUDE_BASELINE else [])
artefacts_ready = (
    all(run.history_path(name).is_file() for name in expected)
    and run.result_file.is_file()
    and {row["model"] for row in rows} == set(expected)
)
is_completed = RUN_TRAINING and artefacts_ready  # smoke test không tính là hoàn thành
marker = run.result_dir / "is_completed.json"
marker.write_text(
    json.dumps(
        {
            "is_completed": is_completed,
            "mode": "full" if RUN_TRAINING else "smoke",
            "artefacts_ready": artefacts_ready,
            "models": expected,
        }
    ),
    encoding="utf-8",
)
print(f"is_completed = {is_completed} ({'full' if RUN_TRAINING else 'smoke'}) -> {marker}")